# Milestone 4b — CNN Feature Re-extraction

Rebuilds `stimulus_bank.pkl` from raw `.mat` files (with the corrected category
assignment) and re-extracts all seven CNN backbones.

Each arch cell is **idempotent**: it skips extraction if the cache already exists,
so you can re-run individual cells after a failure without touching completed arches.

**Run order:** cells must be run top-to-bottom on first pass. After that, any
single arch cell can be re-run in isolation.

**Prerequisites:**
- `pip install tqdm cornet` if not already installed
- Raw `.mat` stimulus files present at `data/ds004194/stimuli/`

In [1]:
import sys, pickle, time
from pathlib import Path

_CANDIDATES = [
    Path('/Users/winstonluk/Documents/NEURON/FinalProject'),
    Path('/sessions/modest-sweet-noether/mnt/FinalProject'),
    Path.cwd().parent,
]
ROOT = next(p for p in _CANDIDATES if (p / 'src' / 'cnn_features.py').exists())
sys.path.insert(0, str(ROOT / 'src'))

from cnn_features import (
    load_brands_stimulus_bank,
    build_and_cache_features,
    load_feature_cache,
)

STIMULI  = ROOT / 'data' / 'ds004194' / 'stimuli'
RESULTS  = ROOT / 'results'
BANK_PKL = RESULTS / 'cache' / 'stimulus_bank.pkl'

print('ROOT   :', ROOT)
print('STIMULI:', STIMULI)
print('exists :', STIMULI.exists())

ROOT   : /Users/winstonluk/Documents/NEURON/FinalProject
STIMULI: /Users/winstonluk/Documents/NEURON/FinalProject/data/ds004194/stimuli
exists : True


## 1. Stimulus bank

Deletes and rebuilds `stimulus_bank.pkl` so the corrected category assignment
(`cat[ti-1]` instead of zipping `cat` with `trialindex`) takes effect.
Confirm **48 images per category** before proceeding.

In [2]:
if BANK_PKL.exists():
    print(f'Removing stale bank: {BANK_PKL}')
    BANK_PKL.unlink()

t0 = time.perf_counter()
bank = load_brands_stimulus_bank(STIMULI)
elapsed = time.perf_counter() - t0

BANK_PKL.parent.mkdir(parents=True, exist_ok=True)
with open(BANK_PKL, 'wb') as f:
    pickle.dump(bank, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Built in {elapsed:.1f}s  —  {len(bank)} unique images')
print(bank.metadata.groupby('category').size().to_string())
print('\nimages shape:', bank.images.shape, bank.images.dtype)

Removing stale bank: /Users/winstonluk/Documents/NEURON/FinalProject/results/cache/stimulus_bank.pkl
Built in 17.4s  —  288 unique images
category
bodies       48
buildings    48
faces        48
objects      48
scenes       48
scrambled    48

images shape: (288, 3, 568, 568) uint8


## 2. ResNet50

In [3]:
out = RESULTS / 'cnn_features_resnet50.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='resnet50', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_resnet50.pkl
  conv1                          (288, 64)
  layer1                         (288, 256)
  layer2                         (288, 512)
  layer3                         (288, 1024)
  layer4                         (288, 2048)
  avgpool                        (288, 2048)


## 3. AlexNet

In [4]:
out = RESULTS / 'cnn_features_alexnet.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='alexnet', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_alexnet.pkl
  features.0                     (288, 64)
  features.3                     (288, 192)
  features.6                     (288, 384)
  features.8                     (288, 256)
  features.10                    (288, 256)
  classifier.4                   (288, 4096)


## 4. VGG16

In [5]:
out = RESULTS / 'cnn_features_vgg16.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='vgg16', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_vgg16.pkl
  features.3                     (288, 64)
  features.8                     (288, 128)
  features.15                    (288, 256)
  features.22                    (288, 512)
  features.29                    (288, 512)
  classifier.3                   (288, 4096)


## 5. ConvNeXt-Tiny

In [6]:
out = RESULTS / 'cnn_features_convnext_tiny.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='convnext_tiny', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_convnext_tiny.pkl
  features.1                     (288, 96)
  features.3                     (288, 192)
  features.5                     (288, 384)
  features.7                     (288, 768)
  classifier.2                   (288, 1000)


## 6. DenseNet-121

In [7]:
out = RESULTS / 'cnn_features_densenet121.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='densenet121', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_densenet121.pkl
  features.denseblock1           (288, 256)
  features.denseblock2           (288, 512)
  features.denseblock3           (288, 1024)
  features.denseblock4           (288, 1024)
  classifier                     (288, 1000)


## 7. Swin-T

In [8]:
out = RESULTS / 'cnn_features_swin_t.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='swin_t', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

Cache exists — loading cnn_features_swin_t.pkl
  features.1                     (288, 96)
  features.3                     (288, 192)
  features.5                     (288, 384)
  features.7                     (288, 768)
  head                           (288, 1000)


## 8. CORnet-S

Requires `pip install cornet`. Recurrent areas (V2, V4, IT) produce per-step
features stored as `area.output@step_N` — e.g. V4 yields 4 keys corresponding
to 4 recurrent passes, each `(288, 512)`.

In [9]:
try:
    import cornet  # noqa: F401
except ImportError:
    raise ImportError('cornet not installed — run: pip install cornet')

out = RESULTS / 'cnn_features_cornet_s.pkl'
if out.exists():
    print(f'Cache exists — loading {out.name}')
    cache = load_feature_cache(out)
else:
    cache = build_and_cache_features(bank, out_path=out, arch='cornet_s', progress=True)
    print(f'Saved → {out}')

for ln, shape in cache.shapes().items():
    print(f'  {ln:30s} {shape}')

cornet_s: 100%|██████████| 18/18 [01:06<00:00,  3.72s/batch]

Saved → /Users/winstonluk/Documents/NEURON/FinalProject/results/cnn_features_cornet_s.pkl
  V1.output                      (288, 64)
  V2.output@step_0               (288, 128)
  V2.output@step_1               (288, 128)
  V4.output@step_0               (288, 256)
  V4.output@step_1               (288, 256)
  V4.output@step_2               (288, 256)
  V4.output@step_3               (288, 256)
  IT.output@step_0               (288, 512)
  IT.output@step_1               (288, 512)


## 9. Verification — all caches

In [10]:
import pandas as pd

archs = ['resnet50', 'alexnet', 'vgg16', 'convnext_tiny', 'densenet121', 'swin_t', 'cornet_s']
rows = []
for arch in archs:
    p = RESULTS / f'cnn_features_{arch}.pkl'
    if not p.exists():
        rows.append({'arch': arch, 'status': 'MISSING', 'n_layers': '', 'n_images': '', 'size_mb': ''})
        continue
    t0 = time.perf_counter()
    c = load_feature_cache(p)
    dt = time.perf_counter() - t0
    rows.append({
        'arch': arch,
        'status': 'OK',
        'n_layers': len(c.layers),
        'n_images': c.n_images,
        'size_mb': f'{p.stat().st_size / 1e6:.1f}',
        'load_ms': f'{dt*1000:.0f}',
    })

pd.DataFrame(rows).set_index('arch')

,status,n_layers,n_images,size_mb,load_ms
arch,,,,,
resnet50,OK,6,288,6.9,8
alexnet,OK,6,288,6.1,4
vgg16,OK,6,288,6.4,3
convnext_tiny,OK,5,288,2.8,2
densenet121,OK,5,288,4.4,3
swin_t,OK,5,288,2.8,4
cornet_s,OK,9,288,2.7,3
